In [97]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import pandas as pd
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import numpy as np
# keras.layers.Dense
# keras.models.Sequential
# tf.keras.optimizers.RMSprop(0.001)

In [98]:
tf.keras.utils.set_random_seed(42)
dataset = pd.read_csv("sonar.csv", header = None)
dataset.head()

,0,1,2,3,4,5,6,7,8,9,...,51,52,53,54,55,56,57,58,59,60
0,0.0200,0.0371,0.0428,0.0207,0.0954,0.0986,0.1539,0.1601,0.3109,0.2111,...,0.0027,0.0065,0.0159,0.0072,0.0167,0.0180,0.0084,0.0090,0.0032,R
1,0.0453,0.0523,0.0843,0.0689,0.1183,0.2583,0.2156,0.3481,0.3337,0.2872,...,0.0084,0.0089,0.0048,0.0094,0.0191,0.0140,0.0049,0.0052,0.0044,R
2,0.0262,0.0582,0.1099,0.1083,0.0974,0.2280,0.2431,0.3771,0.5598,0.6194,...,0.0232,0.0166,0.0095,0.0180,0.0244,0.0316,0.0164,0.0095,0.0078,R
3,0.0100,0.0171,0.0623,0.0205,0.0205,0.0368,0.1098,0.1276,0.0598,0.1264,...,0.0121,0.0036,0.0150,0.0085,0.0073,0.0050,0.0044,0.0040,0.0117,R
4,0.0762,0.0666,0.0481,0.0394,0.0590,0.0649,0.1209,0.2467,0.3564,0.4459,...,0.0031,0.0054,0.0105,0.0110,0.0015,0.0072,0.0048,0.0107,0.0094,R


In [99]:
train_dataset, test_dataset = train_test_split(dataset, test_size = 0.2, random_state = 42, stratify = dataset[60])
train_dataset, validation_dataset = train_test_split(train_dataset, test_size = 0.2, random_state = 42, stratify = train_dataset[60])

In [100]:
# NEURONSKA MREZA RADI SAMO SA BROJEVIMA -> NE ZABORAVLJAJ MAPIRANJE
train_labels = train_dataset.pop(60).map({"R": 0, "M": 1})
validation_labels = validation_dataset.pop(60).map({"R": 0, "M": 1})
test_labels = test_dataset.pop(60).map({"R": 0, "M": 1})

In [101]:
# ONE HOT ENCODING

train_labels_cat = keras.utils.to_categorical(train_labels)
validation_labels_cat = keras.utils.to_categorical(validation_labels)
test_labels_cat = keras.utils.to_categorical(test_labels)

In [102]:
train_dataset = train_dataset.astype('float32')
validation_dataset = validation_dataset.astype('float32')
test_dataset = test_dataset.astype('float32')

In [103]:
def build_model(learning_rate):
  model = keras.Sequential([
      layers.Dense(32, activation = 'relu', input_shape = [len(train_dataset.keys())]),
      layers.Dense(16, activation = 'relu'),
      layers.Dense(2, activation = 'softmax')
  ])

  model.compile(optimizer = tf.keras.optimizers.RMSprop(learning_rate), loss = tf.keras.losses.CategoricalCrossentropy(), metrics = ["accuracy"])

  return model

In [104]:
learning_rates = [0.1, 0.01, 0.001, 0.0001]

In [105]:
validation_results = []

for learning_rate in learning_rates:
  tf.keras.backend.clear_session()
  tf.keras.utils.set_random_seed(42)

  model = build_model(learning_rate)
  history = model.fit(train_dataset, train_labels_cat, epochs = 100, batch_size = 16, validation_data = (validation_dataset, validation_labels_cat), verbose = 0)

  best_validation_accuracy = max(history.history["val_accuracy"])
  best_epoch = history.history["val_accuracy"].index(best_validation_accuracy) + 1
  validation_results.append({
      "learning_rate": learning_rate,
      "best_validation_accuracy": best_validation_accuracy,
      "best_epoch": best_epoch
  })

result_df = pd.DataFrame(validation_results)
result_df

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/usr

,learning_rate,best_validation_accuracy,best_epoch
0,0.1000,0.529412,1
1,0.0100,0.911765,58
2,0.0010,0.852941,60
3,0.0001,0.588235,86


In [106]:
best_result = sorted(validation_results, key = lambda result: (-result["best_validation_accuracy"], result["learning_rate"]))[0]
best_learning_rate = best_result["learning_rate"]

In [108]:
tf.keras.backend.clear_session()
tf.keras.utils.set_random_seed(42)

train_dataset_full = pd.concat([train_dataset, validation_dataset])
train_labels_cat_full = np.concatenate([train_labels_cat, validation_labels_cat])

model = build_model(best_learning_rate)
history = model.fit(train_dataset_full, train_labels_cat_full, epochs = 100, batch_size = 16, verbose = 0)

test_loss, test_accuracy = model.evaluate(test_dataset, test_labels_cat, verbose = 0)
print("test loss: ", test_loss)
print("test accuracy: ", test_accuracy)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


test loss:  1.4035933017730713
test accuracy:  0.738095223903656
